# Lab 18 — Train a Churn Model on the GPU

You have already run a **Spark job** that turned 849 MB of raw CSV into 232 MB of Parquet.
This notebook does everything that comes after: it builds the training data, trains a model
on the GPU, and registers it in MLflow under your name.

---

## What you will do

| Part | What happens | Concept |
|---|---|---|
| 1 | Set your student number, check the GPU | environment |
| 2 | Look at what Spark produced | Parquet, partitions |
| 3 | Same job on CPU, then GPU — **timed** | **RAPIDS / cuDF** |
| 4 | Fetch the answer from a different system | the label |
| 5 | Join behaviour to identity on the GPU | building a training table |
| 6 | Train a model | XGBoost on GPU |
| 7 | See which columns mattered | feature importance |
| 8 | Register the model under your name | MLflow |

## Before you start

- ✅ Your Spark job finished — you should have a `curated/student-NN/` folder
- ✅ You are on a **GPU** notebook
- ✅ You know your **student number**

## How to run this notebook

> **Run the cells in order, from the top.** Use `Shift + Enter`, or
> **Kernel → Restart & Run All**.
>
> ⚠️ Skipping ahead will fail — later cells use variables defined earlier. If you get a
> `NameError`, you skipped a cell. Restart and run from the top.


---
# Part 1 — Setup

## 1.1 — Install what this notebook needs

**▶ Run this cell first.** A few seconds, safe to re-run.

It installs in **two steps, and the order matters**:

1. `psycopg2`, `xgboost`, `scikit-learn`, `mlflow` — letting pip resolve them freely
2. NumPy and pandas — forced afterwards to the versions everything else needs

Why two steps: mlflow wants `pandas<3`, while the GPU dataframe library wants
`pandas>=3`. Ask pip to satisfy both in one command and it hunts backwards through
mlflow releases looking for peace, eventually installing a version from 2022 with
broken metadata — and leaves your environment worse than it found it. Installing
mlflow first and correcting NumPy/pandas afterwards sidesteps the argument entirely.
mlflow prints a version warning and works fine.

> **cuDF is deliberately NOT installed here.** It is a 1–2 GB download and needs a
> *kernel restart*, which would break "run every cell from the top". Either it is
> already in your image — the next cell tells you — or the notebook falls back to the
> CPU and everything still works, minus the speed comparison.

In [ ]:
import sys, subprocess, importlib.util

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# STEP 1 — let these resolve freely. Pinning numpy/pandas here makes pip backtrack
# through mlflow releases and install a broken 2022 version.
print("installing packages ...")
pip("psycopg2-binary", "xgboost", "scikit-learn", "mlflow")

# STEP 2 — now force the numpy/pandas pair. Last install wins.
# cuDF needs pandas 3.x; without cuDF, pandas 2.2.3+ is the compatible choice.
has_cudf = importlib.util.find_spec("cudf") is not None
if has_cudf:
    print("cuDF detected — pinning pandas 3.x to match it")
    pip("numpy>=2,<3", "pandas>=3.0,<3.0.4")
else:
    print("no cuDF — pinning the standard numpy 2 / pandas 2.2+ pair")
    pip("numpy>=2,<3", "pandas>=2.2.3")

import numpy, pandas
print(f"\nnumpy  {numpy.__version__}")
print(f"pandas {pandas.__version__}")

if int(numpy.__version__.split('.')[0]) < 2:
    print("\n*** numpy is below 2 — something downgraded it. Re-run this cell, "
          "then restart the kernel. ***")
else:
    print("\ndone — dependencies ready")

## 1.2 — Your student number

**▶ Edit the number below to your own, then run the cell.**

This is the only thing in the notebook you change. Everything else — which folder you read,
what your experiment is called, what your model is named — is built from this number, so
everyone in the room works independently without collisions.

*(Note the absolute path. A notebook's working directory is your home folder, not wherever
you think it is, so we never use relative paths.)*

In [ ]:
import os

STUDENT_ID = 0                       # ←←← CHANGE THIS TO YOUR NUMBER

SID     = f'{STUDENT_ID:02d}'        # zero-padded: 0 -> '00', 7 -> '07'
LAB     = os.path.expanduser('~/shared/data-engineering-lab')
CURATED = f'{LAB}/curated/student-{SID}/watch_events'
CUTOFF  = '2026-05-31'               # we use the LAST 30 DAYS of history as features

print(f'You are student {SID}')
print(f'Reading from : {CURATED}')
print(f'Feature window: events on or after {CUTOFF}')

assert os.path.isdir(CURATED), (
    f'\nNo curated data at {CURATED}\n'
    f'Either your Spark job has not finished, or STUDENT_ID is wrong.')
print('\n✅ Your curated data is there — carry on.')

## 1.3 — GPU check, and a courtesy to your neighbours

**▶ Run this cell. It must run before any other cell that touches the GPU.**

You are sharing a physical GPU with other people. Sharing splits the *compute*, but **not the
memory** — so if one notebook grabs all the GPU memory, everyone else on that card crashes.

The cell below caps this notebook at 1.5 GB of GPU memory. Everything we do fits easily inside
that. If you restart the kernel later, run this cell first again.

**What is cuDF?** It is a dataframe library that looks *exactly* like pandas but runs on the
GPU. Same method names, same syntax — different hardware. It is part of **RAPIDS**, NVIDIA's
suite of GPU data-science libraries. You will see the difference it makes in Part 3.

👀 **Look for:** the GPU name, and `cuDF ... available`. If cuDF is missing the notebook still
works, it just falls back to the CPU and you miss the speed comparison.

In [ ]:
import subprocess

try:
    import rmm                                    # RAPIDS memory manager
    rmm.reinitialize(pool_allocator=True,
                     initial_pool_size='256MB',
                     maximum_pool_size='1500MB')  # ← the cap that protects everyone else
    import cudf
    HAS_GPU_DF = True
    print(f'✅ cuDF {cudf.__version__} available — GPU memory capped at 1.5 GB')
except Exception as e:
    HAS_GPU_DF, cudf = False, None
    print('⚠️  cuDF not available — the notebook will use the CPU instead.')
    print('   (', str(e)[:70], ')')

print()
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.used',
                      '--format=csv'], capture_output=True, text=True).stdout)

---
# Part 2 — What did Spark actually produce?

## The format change, and why it matters

Your Spark job did **not** change the data. Same 20,010,929 rows went in and came out. What it
changed was the **format**:

```
CSV      stored ROW by row, as text
         168654,5,478,2026-04-11,4.1,web,1
         168655,8,1902,2026-04-11,54.9,tablet,1

Parquet  stored COLUMN by column, in binary
         [all the event_ids] [all the subscriber_ids] [all the watch_minutes] ...
```

Two consequences, and you will feel both in Part 3:

1. **It is 3.66× smaller** — 849 MB became 232 MB. A column of repeated values
   (`tv, tv, mobile, tv`) compresses beautifully when those values sit together.
2. **You can read one column without reading the others.** We need 4 of the 7 columns.
   Parquet reads just those 4. CSV would have to read every character of every line to find
   them.

This is called **columnar storage**, and it is why analytics systems everywhere use Parquet
rather than CSV.

## Partitions — only read what you need

Spark wrote one folder per day: `dt=2026-01-01`, `dt=2026-01-02`, and so on — 180 of them.

We only want the **last 30 days**, so we read 30 folders and never open the other 150. That is
**partition pruning**: a real speed-up that comes purely from how the files were named.

**▶ Run the cell.** 👀 **Look for:** 180 total, 30 kept.

In [ ]:
# Guard: this needs the Config cell in Part 1. Running cells out of order is the most
# common mistake, and a bare NameError does not explain it.
for _v in ('SID', 'CURATED', 'CUTOFF'):
    if _v not in globals():
        raise RuntimeError(f"'{_v}' is not defined — run Part 1 first "
                           "(Kernel -> Restart & Run All).")

import glob, datetime

cutoff   = datetime.date.fromisoformat(CUTOFF)
day_dirs = sorted(glob.glob(f'{CURATED}/dt=*'))
keep     = [d for d in day_dirs
            if datetime.date.fromisoformat(d.split('dt=')[1]) >= cutoff]
files    = [f for d in keep for f in glob.glob(f'{d}/*.parquet')]

print(f'day-folders Spark wrote   : {len(day_dirs)}')
print(f'day-folders we will read  : {len(keep)}   ← the other '
      f'{len(day_dirs)-len(keep)} are never opened')
print(f'parquet files to read     : {len(files)}')

---
# Part 3 — The RAPIDS moment: CPU vs GPU

## What we are about to compute

The raw data has **one row per viewing session** — roughly 100 rows per person. A model needs
**one row per person**. So for each subscriber we roll up their last 30 days:

| From (many rows) | To (one row) |
|---|---|
| every viewing | `minutes_30d` — total minutes watched |
| | `sessions_30d` — how many times they watched |
| | `completion_rate` — fraction they finished |
| | `distinct_titles_30d` — how many different titles |

In SQL this is `GROUP BY`. In pandas and cuDF it is `.groupby()`. We call it **the squash**.

These four columns are **features** — facts about a person, computed from their behaviour.
They do not exist anywhere in your data. You are inventing them. That invention is most of
applied machine learning.

---

## ✋ Before you run the next two cells

We will do the **identical** calculation twice — once on the CPU with pandas, once on the GPU
with cuDF. The code is the same function both times.

> **Write down your guess: how many times faster will the GPU be?**

**▶ Run the CPU cell first.**

In [ ]:
import pandas as pd, time

COLS = ['subscriber_id', 'watch_minutes', 'completed', 'content_id']

def squash(df):
    """Many rows per person -> one row per person.
    This exact function runs on BOTH pandas and cuDF — same API, different hardware."""
    g = df.groupby('subscriber_id').agg(
        {'watch_minutes': ['sum', 'count'],
         'completed': 'mean',
         'content_id': 'nunique'})
    g.columns = ['minutes_30d', 'sessions_30d', 'completion_rate', 'distinct_titles_30d']
    return g.reset_index()

print('Reading with pandas (CPU) ...')
t0 = time.time()
pdf = pd.concat([pd.read_parquet(f, columns=COLS) for f in files], ignore_index=True)
t_read_cpu = time.time() - t0

print('Squashing with pandas (CPU) ...')
t0 = time.time()
usage_cpu = squash(pdf)
t_agg_cpu = time.time() - t0

print(f'\n  rows read       : {len(pdf):,}')
print(f'  people out      : {len(usage_cpu):,}')
print(f'  CPU read time   : {t_read_cpu:6.2f} s')
print(f'  CPU squash time : {t_agg_cpu:6.2f} s')
usage_cpu.head()

## Now the same thing on the GPU

**▶ Run this cell.** Nothing about the calculation changes — `squash()` is the same function.
Only the library differs: `cudf.read_parquet` instead of `pd.read_parquet`.

👀 **Look for:** the speed-up line at the bottom. Compare it to your guess.

*(The GPU read includes a one-time start-up cost, so the read number is pessimistic. The
squash number is the honest comparison.)*

In [ ]:
if HAS_GPU_DF:
    print('Reading with cuDF (GPU) ...')
    t0 = time.time()
    gdf = cudf.read_parquet(files, columns=COLS)
    t_read_gpu = time.time() - t0

    print('Squashing with cuDF (GPU) ...')
    t0 = time.time()
    usage_gpu = squash(gdf)          # ← the SAME function as the CPU cell
    t_agg_gpu = time.time() - t0

    print(f'\n  rows read       : {len(gdf):,}')
    print(f'  GPU read time   : {t_read_gpu:6.2f} s')
    print(f'  GPU squash time : {t_agg_gpu:6.2f} s')
    print('\n' + '=' * 52)
    print(f'  SQUASH SPEED-UP : {t_agg_cpu / max(t_agg_gpu, 1e-6):5.1f}x'
          f'   (CPU {t_agg_cpu:.2f}s  ->  GPU {t_agg_gpu:.2f}s)')
    print('=' * 52)
    usage = usage_gpu.to_pandas()
else:
    print('cuDF not available — using the CPU result from the previous cell.')
    usage = usage_cpu

print(f'\n{len(usage):,} people, one row each. This is HALF of the training data —')
print('it says what people DID, but not whether they cancelled.')
usage.head()

---
# Part 4 — The answer lives somewhere else

Everything so far has been **behaviour** — what people watched. But the thing we want to
predict, `churned_next_30d`, is **not in the event data at all**.

It lives in a completely different system: a **Postgres database** holding the customer
records — plan, tenure, support tickets, and whether they cancelled.

```
watch_events   →  files on shared storage  →  what people DID       (no answer)
subscribers    →  a Postgres database      →  who people ARE + THE ANSWER
```

**This is the point of the whole exercise.** In any real company, activity logs and customer
records live in different systems owned by different teams. You cannot train anything until
you bring them together.

### Why the answer matters

A model learns by example. You show it thousands of people where you *already know* what
happened, it works out the pattern, and then it can predict for someone new.

> Past exam papers **with the answer key** are your training data. The real exam has no answer
> key — that is prediction. You cannot learn from past papers if nobody wrote the answers down.

`churned_next_30d` is the answer key.

**▶ Run the cell.** 👀 **Look for:** 200,000 rows and a churn rate near 0.12 — about 12% of
people cancelled.

In [ ]:
# psycopg2 is small and shares no dependencies with numpy/pandas/cuDF, so unlike
# cuDF it is safe to install inline — no kernel restart needed. The version pins
# stop pip from quietly downgrading numpy underneath cuDF.
try:
    import psycopg2
except ImportError:
    import sys, subprocess
    print('installing psycopg2-binary ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'psycopg2-binary', 'numpy>=2,<3', 'pandas>=3.0,<3.0.4'],
                   check=False)
    import psycopg2
    print('ok')


PG = dict(host='postgres-data-postgresql.postgres-data.svc.cluster.local',
          port=5432, dbname='app_db',
          user='admin', password='admin',      # read-only account
          connect_timeout=10)

conn = psycopg2.connect(**PG)
subs_pd = pd.read_sql('''
    SELECT subscriber_id, plan, country, payment_method, monthly_price,
           support_tickets_90d, tenure_months, churned_next_30d
    FROM   public.subscribers''', conn)
conn.close()

print(f'subscribers : {len(subs_pd):,} rows')
print(f'churn rate  : {subs_pd.churned_next_30d.mean():.4f}   '
      f'({subs_pd.churned_next_30d.sum():,} people cancelled)')
subs_pd.head()

---
# Part 5 — Join the two halves, on the GPU

200,000 rows of behaviour meet 200,000 rows of identity. `subscriber_id` is the link.

## One subtlety worth understanding

We use a **LEFT join** starting from `subscribers`, not an inner join.

Why: somebody who watched **nothing** in the last 30 days has no row in the behaviour table.
An inner join would quietly drop them — and those are exactly the people most likely to have
cancelled. So we keep every subscriber and fill their missing activity with **zeros**.

> **Absence of a record is itself information.** Dropping those rows would delete the
> strongest signal in the dataset, and the code would look perfectly correct.

**▶ Run the cell.** 👀 **Look for:** the two-row table at the bottom. People who churned watch
far fewer minutes than people who stayed. **That gap is what the model will learn.**

In [ ]:
def to_gpu(pdf):
    """pandas -> cuDF. The function name moved between cuDF versions, so try both."""
    return cudf.from_pandas(pdf) if hasattr(cudf, 'from_pandas') else cudf.DataFrame(pdf)

FILL = {'minutes_30d': 0.0, 'sessions_30d': 0,
        'completion_rate': 0.0, 'distinct_titles_30d': 0}

if HAS_GPU_DF:
    feat_df = (to_gpu(subs_pd)
               .merge(to_gpu(usage), on='subscriber_id', how='left')
               .fillna(FILL)
               .to_pandas())
    where = 'GPU'
else:
    feat_df = (subs_pd.merge(usage, on='subscriber_id', how='left').fillna(FILL))
    where = 'CPU'

print(f'Training table ({where} join): {len(feat_df):,} rows x {feat_df.shape[1]} columns\n')

summary = (feat_df.groupby('churned_next_30d')
                  .agg(people=('subscriber_id', 'count'),
                       avg_minutes_30d=('minutes_30d', 'mean'),
                       avg_sessions=('sessions_30d', 'mean')).round(1))
summary.index = ['stayed (0)', 'churned (1)']
print(summary)
print('\n👉 People who cancelled watched far less. That gap IS the model.')

---
# Part 6 — Train the model

We use **XGBoost**, a gradient-boosted tree model — the standard choice for tabular data like
this.

Two things happen first:

1. **One-hot encoding.** A model needs numbers, not words. `plan` becomes three 0/1 columns:
   `plan_basic`, `plan_standard`, `plan_premium`.
2. **Train/test split.** We train on 75% and keep 25% hidden. Scoring on data the model has
   already seen would tell us nothing — it could simply memorise.

`device='cuda'` puts the training on the GPU. One keyword; no rewrite.

## Reading the score

**AUC** measures how well the model separates the two groups. Pick a churner and a non-churner
at random: AUC is the probability the model scores the churner higher.

| AUC | Meaning |
|---|---|
| 0.5 | Useless — coin flip |
| 0.7 | Some signal |
| **~0.87** | **Good — what you should get** |
| 1.0 | Perfect. Be suspicious: usually a leak |

**▶ Run the cell.** 👀 **Look for:** AUC around 0.87.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

y = feat_df['churned_next_30d']
X = feat_df.drop(columns=['subscriber_id', 'churned_next_30d'])
X = pd.get_dummies(X, columns=['plan', 'country', 'payment_method'])  # words -> numbers

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

print(f'training on {len(X_tr):,} people, testing on {len(X_te):,} held-out people')
print(f'{X.shape[1]} features after one-hot encoding\n')

def make(device):
    return xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                             subsample=0.9, tree_method='hist', device=device,
                             eval_metric='auc')

try:
    model = make('cuda'); model.fit(X_tr, y_tr); dev = 'GPU'
except Exception as e:
    print('GPU training unavailable, falling back to CPU:', str(e)[:70])
    model = make('cpu');  model.fit(X_tr, y_tr); dev = 'CPU'

model.set_params(device='cpu')      # inference on host data; avoids a device-mismatch warning
auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

print(f'\n{"=" * 46}')
print(f'  TEST AUC: {auc:.4f}     ({dev} training)')
print('=' * 46)
print('\n📝 Write this number down — it goes on the leaderboard.')

---
# Part 7 — Which columns actually mattered?

Now the interesting part. Two things to look for, and the second is the real lesson.

### 1. Similar columns share the credit
`minutes_30d`, `sessions_30d` and `distinct_titles_30d` all measure the same underlying thing —
how much someone watches. A tree model picks one or two and largely ignores the rest, so do not
be surprised if `minutes_30d` ranks low. That does **not** mean watch-time is irrelevant; you
saw in Part 5 that churners watch a third as much. Importance shows what the model *used*, not
what the world *contains*.

### 2. Useless columns do not score zero — they score *identically*
`country` was put into this dataset as **deliberate noise**. It predicts nothing. But every
column picks up a little importance by chance, so it will not be zero.

The giveaway is the **spread**. Watch the six `country_*` values land within a hair of each
other while the real features differ by an order of magnitude.

> **Real features disagree with each other. Noise features are indistinguishable.**
> That tight little cluster is the fingerprint of a column carrying nothing — and it is how you
> spot a useless column in practice.

**▶ Run the cell.**

In [ ]:
imp = (pd.Series(model.feature_importances_, index=X.columns)
         .sort_values(ascending=False))

print('TOP 8 FEATURES')
print(imp.head(8).to_string())

noise = imp[[c for c in imp.index if c.startswith('country_')]]
real  = imp[[c for c in imp.index if not c.startswith('country_')]]

print('\nTHE PLANTED NOISE (country)')
print(noise.to_string())

print('\n' + '=' * 58)
print(f'  noise columns : spread {noise.max()-noise.min():.4f}  '
      f'(from {noise.min():.4f} to {noise.max():.4f})')
print(f'  real  columns : spread {real.max()-real.min():.4f}  '
      f'(from {real.min():.4f} to {real.max():.4f})')
print('=' * 58)
print('Real features disagree with each other. Noise features do not.')

---
# Part 8 — Register the model in MLflow

A trained model sitting in a notebook is worthless to everyone else. **MLflow** is the record:
it stores what was trained, with which settings, by whom, and what score it got.

That record is what makes a model:

- **findable** — a colleague can locate your model without asking you
- **comparable** — v1 against v2, side by side
- **auditable** — every parameter and every promotion is logged
- **deployable** — serving reads from the registry, not from your laptop

Your model is registered as **`student-NN-churn`**, so it is yours and appears on the
leaderboard.

**▶ Run the cell.** 👀 **Look for:** the run URL. Open it and look at your parameters, your
metric, and the registered model.

In [ ]:
import mlflow

try:
    uri = os.environ.get('MLFLOW_TRACKING_URI',
                         'http://mlflow.mlflow.svc.cluster.local:5000')
    tok = '/etc/secrets/ezua/.auth_token'
    if os.path.exists(tok):
        os.environ['MLFLOW_TRACKING_TOKEN'] = open(tok).read().strip()
    mlflow.set_tracking_uri(uri)
    mlflow.set_experiment(f'student-{SID}-churn')

    with mlflow.start_run(run_name=f'student-{SID}') as run:
        mlflow.log_params({'n_estimators': 300, 'max_depth': 6,
                           'learning_rate': 0.1, 'subsample': 0.9})
        mlflow.log_metric('auc', auc)
        mlflow.set_tags({'student_id': SID, 'dataset': 'streaming-churn',
                         'rows_trained_on': len(X_tr)})
        mlflow.xgboost.log_model(model, 'model',
                                 registered_model_name=f'student-{SID}-churn')
    print(f'\n✅ Registered as: student-{SID}-churn')
    print(f'   AUC {auc:.4f}')
except Exception as e:
    print('MLflow logging skipped:', str(e)[:200])
    print('Your model and AUC above are still valid — only the registry step failed.')

---
# Optional — beat your own score

You now have a baseline. Try to improve it.

Go back to **Part 6**, change one or two numbers, and re-run Parts 6 → 8:

```python
max_depth      = 8      # deeper trees (was 6)
n_estimators   = 500    # more trees (was 300)
learning_rate  = 0.05   # smaller steps (was 0.1)
```

Re-running Part 8 creates **version 2** of your model in MLflow. Open the registry and compare
v1 and v2 side by side — that is what model versioning is *for*.

⚠️ Do not expect miracles. With one dominant signal, hyperparameters move AUC by a fraction of
a percent. Discovering that is itself worth knowing: **better features beat better
hyperparameters, almost always.**

---
# What you just did

```
  20,010,929 event rows        (what people watched)
          │  cuDF on the GPU — the squash
          ▼
     200,000 people
          │  + the answer, from a Postgres database
          ▼
     200,000 x 12 training table
          │  XGBoost on the GPU
          ▼
     a model, AUC ~0.87, registered under your name
```

## The ideas, not the tools

| Idea | Where you saw it |
|---|---|
| **Columnar storage** | Parquet — 3.66× smaller, reads only the columns you ask for |
| **Partition pruning** | reading 30 day-folders instead of 180 |
| **Aggregation / feature engineering** | the squash — you invented `minutes_30d` |
| **Hardware acceleration** | cuDF — same API, different hardware |
| **Joining across systems** | files + a live database in one training table |
| **Labels** | you can only learn from history where the answer was recorded |
| **Absence as information** | the LEFT join, and filling with zeros |
| **Model registry** | MLflow — findable, comparable, auditable, deployable |

---

**The hard part of machine learning was getting the table right.** Everything after that was
about ten lines of code.